# Module 06: Final Data Science Project
## Palmer Penguins: Exploratory Analysis and Species Classification

This end-to-end project uses a public penguin measurements dataset to explore patterns and train a machine learning model that predicts species from physical measurements. It covers data loading, cleaning, visualization, model training, test-set evaluation, and project limitations.

## Project goals
1. Load and inspect a real-world public dataset.
2. Clean duplicate rows and handle missing measurements appropriately.
3. Visualize how measurements differ by species.
4. Train a species classifier and evaluate it on held-out test data.

**Prediction target:** penguin species. **Features:** bill length, bill depth, flipper length, and body mass. We do not use island or sex as model features.

## 1. Import libraries and load data
The CSV is read from the public `palmerpenguins` repository. Run this notebook in a Colab runtime with internet access.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='Set2')
DATA_URL = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
df = pd.read_csv(DATA_URL, na_values=['NA', ''])
print('Original dataset shape:', df.shape)
display(df.head())

## 2. Explore and clean the dataset
Inspect data types, missing values, and duplicate rows. Remove duplicate records and rows without a target label. Keep missing feature values for now: the model pipeline will impute them using statistics learned from training data only.

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', int(df.duplicated().sum()))

clean_df = df.drop_duplicates().dropna(subset=['species']).copy()
clean_df['species'] = clean_df['species'].astype(str)
print('\nShape after removing duplicates and missing target labels:', clean_df.shape)
print('Rows with at least one missing measurement:', int(clean_df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].isna().any(axis=1).sum()))

## 3. Exploratory analysis
Summarize sample counts and average measurements for each species. The data is observational, so these comparisons describe the sample rather than prove causes.

In [ ]:
measurement_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
species_summary = clean_df.groupby('species')[measurement_features].agg(['count', 'mean']).round(1)
print('Measurement summaries by species:')
display(species_summary)
print('\nSpecies counts:')
display(clean_df['species'].value_counts())

## 4. Visualize the data
The charts compare species counts, body mass distributions, and relationships between physical measurements.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=clean_df, x='species', order=clean_df['species'].value_counts().index, ax=axes[0])
axes[0].set_title('Observations by Species')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')
sns.boxplot(data=clean_df, x='species', y='body_mass_g', ax=axes[1])
axes[1].set_title('Body Mass by Species')
axes[1].set_xlabel('Species')
axes[1].set_ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=clean_df, x='flipper_length_mm', y='body_mass_g', hue='species', s=75, alpha=0.85)
plt.title('Flipper Length vs Body Mass by Species')
plt.xlabel('Flipper length (mm)')
plt.ylabel('Body mass (g)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
sns.scatterplot(data=clean_df, x='bill_length_mm', y='bill_depth_mm', hue='species', s=75, alpha=0.85)
plt.title('Bill Length vs Bill Depth by Species')
plt.xlabel('Bill length (mm)')
plt.ylabel('Bill depth (mm)')
plt.tight_layout()
plt.show()

## 5. Prepare training and test sets
Use an 80/20 stratified split so every species is represented proportionally. The held-out test data is not used to fit the imputer, scaler, or classifier.

In [ ]:
X = clean_df[measurement_features]
y = clean_df['species']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')
print('Training species proportions:')
print(y_train.value_counts(normalize=True).round(3))

## 6. Train a baseline and a machine learning model
The baseline always predicts the most common species. The Logistic Regression pipeline imputes missing measurements with training-set medians, scales the measurements, then trains the classifier. Keeping preprocessing inside the pipeline helps prevent data leakage.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ('measurements', numeric_preprocessor, measurement_features)
])
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
print('Baseline and Logistic Regression model training complete.')

## 7. Evaluate on the held-out test data
Report accuracy and macro F1 for the baseline and trained model, then inspect class-level precision, recall, F1, and the confusion matrix. These test metrics estimate performance for this split; they are not a guarantee of future performance.

In [ ]:
predictions = model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
model_accuracy = accuracy_score(y_test, predictions)
model_macro_f1 = f1_score(y_test, predictions, average='macro')
print(f'Baseline accuracy: {baseline_accuracy:.3f}')
print(f'Logistic Regression accuracy: {model_accuracy:.3f}')
print(f'Logistic Regression macro F1: {model_macro_f1:.3f}')
print('\nClassification report:')
print(classification_report(y_test, predictions, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, labels=model.classes_, cmap='Blues', values_format='d'
)
plt.title('Penguin Species Classifier: Test Set')
plt.tight_layout()
plt.show()

## 8. Summarize findings
The final cell reports dataset-based patterns alongside test performance. Compare the visualizations with these outputs before writing your portfolio summary.

In [ ]:
mean_mass = clean_df.groupby('species')['body_mass_g'].mean().sort_values(ascending=False)
mean_flipper = clean_df.groupby('species')['flipper_length_mm'].mean().sort_values(ascending=False)
flipper_mass_corr = clean_df[['flipper_length_mm', 'body_mass_g']].corr().iloc[0, 1]
print(f'Highest average body mass: {mean_mass.index[0]} ({mean_mass.iloc[0]:.0f} g)')
print(f'Highest average flipper length: {mean_flipper.index[0]} ({mean_flipper.iloc[0]:.1f} mm)')
print(f'Flipper length/body mass correlation: {flipper_mass_corr:.2f}')
print(f'Test accuracy improvement over baseline: {model_accuracy - baseline_accuracy:+.3f}')
print('Interpretation: physical measurements show visible differences among species in this dataset. The classifier captures some of these patterns, but its performance should be checked on other samples before real-world use.')

## Conclusion and limitations
This project completes an end-to-end workflow: public data loading, quality checks, cleaning, exploratory visualizations, feature selection, train/test splitting, preprocessing, model training, and evaluation. The dataset is observational, limited in size and geographic scope, and the test result comes from one split. The model is for learning and should not be treated as a scientific or conservation tool.